# Similar assortativity, different local exposure

This notebook reproduces the within-model matched-network analysis in main-text
Fig. 4 and its SI robustness analysis (Fig. S4 and Table S10). Networks generated
under different configurations are paired when their vote assortativity differs by
less than 0.02, with 0.05 used as the SI sensitivity threshold. Nomination-network
matches use all runs; reciprocal-network matches use only runs satisfying all five
structural plausibility criteria.


In [ ]:
from pathlib import Path
import pickle

import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name == "code":
    BASE_DIR = BASE_DIR.parent

DATA_DIR = BASE_DIR / "outputs" / "data"
FIGURE_DIR = BASE_DIR / "outputs" / "figures"
OUT_DIR = FIGURE_DIR / "structure_exposure_link"
IMAGE_DIR = BASE_DIR / "code" / "images"
FONT_DIR = BASE_DIR / "font"
OUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

for font_path in sorted(FONT_DIR.glob("Barlow-*.ttf")):
    font_manager.fontManager.addfont(str(font_path))
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "font.family": "Barlow",
    "font.weight": "normal",
    "axes.labelweight": "normal",
    "axes.titleweight": "normal",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "figure.dpi": 110,
})
pd.set_option("display.max_columns", 100)


In [ ]:
GRAPH_INPUTS = {
    "directed_out": DATA_DIR / "graphs_sample_100_v3_directed.pkl",
    "mutual": DATA_DIR / "graphs_sample_100_v3_mutual.pkl",
}
EXPOSURE_CSV = DATA_DIR / "local_evaluation_exposure_long_sample_100_v3_by_network_view.csv"
PLAUSIBILITY_CSV = DATA_DIR / "structural_plausibility_sample_100_v3_mutual.csv"

NETWORK_VIEWS = ["directed_out", "mutual"]
MODEL_LABELS = {
    "llama-31-8b-instruct": "Llama-3.1 8B",
    "mistral-7b-instruct-v03": "Mistral 7B",
    "qwen25-7b-instruct": "Qwen2.5 7B",
    "meta_llama_31_70b_instruct_awq_int4": "Llama-3.1 70B AWQ",
    "qwen25_72b_instruct_awq": "Qwen2.5 72B AWQ",
}
MODEL_ORDER = [
    "Llama-3.1 8B", "Mistral 7B", "Qwen2.5 7B",
    "Llama-3.1 70B AWQ", "Qwen2.5 72B AWQ",
]
MODEL_COLORS = {
    "Llama-3.1 8B": "#793B56",
    "Mistral 7B": "#B45791",
    "Qwen2.5 7B": "#EF895F",
    "Llama-3.1 70B AWQ": "#86CEB0",
    "Qwen2.5 72B AWQ": "#078BA8",
}
EXPOSURE_RELATIONS = [
    ("dem", "dem", "Dem → Dem"),
    ("dem", "rep", "Dem → Rep"),
    ("rep", "dem", "Rep → Dem"),
    ("rep", "rep", "Rep → Rep"),
    ("other", "dem", "Other → Dem"),
    ("other", "rep", "Other → Rep"),
]
RELATION_ORDER = [relation for _, _, relation in EXPOSURE_RELATIONS]
RELATION_LABELS = {
    relation: relation.replace(" → ", r" $\rightarrow$ ")
    for relation in RELATION_ORDER
}
ASSORTATIVITY_CALIPER = 0.02
SENSITIVITY_CALIPER = 0.05


## Inputs and run-level eligibility

Notebook 02 supplies the mutual-network plausibility flag. Notebook 03 supplies
persona-level exposure for both representations. The same run-level flag is attached to
the directed-out and mutual versions of a run; exposure errors are not recomputed here.


In [ ]:
exposure = pd.read_csv(EXPOSURE_CSV)
plausibility = pd.read_csv(PLAUSIBILITY_CSV)

required_exposure = {
    "network_view", "model", "model_label", "temperature", "method",
    "attr_label", "attribute_order", "run_number", "run_id", "caseid",
    "ego_party", "party", "llm_cat", "survey_cat",
}
required_plausibility = {"model", "file", "plausible_all_5", "plausible_metric_count"}
assert not required_exposure.difference(exposure.columns)
assert not required_plausibility.difference(plausibility.columns)
assert len(exposure) == 400_000
assert set(exposure["network_view"]) == set(NETWORK_VIEWS)
assert len(plausibility) == 1_000
assert plausibility[["model", "file"]].duplicated().sum() == 0

plausibility_keys = plausibility[
    ["model", "file", "plausible_all_5", "plausible_metric_count"]
].rename(columns={"file": "run_id"})
plausibility_keys["plausible_all_5"] = plausibility_keys["plausible_all_5"].astype(bool)

coverage = (
    plausibility.assign(model_label=plausibility["model"].map(MODEL_LABELS))
    .groupby("model_label", observed=True)
    .agg(
        generated_networks=("file", "size"),
        plausible_networks=("plausible_all_5", "sum"),
    )
    .reindex(MODEL_ORDER)
)
coverage["plausible_share"] = coverage["plausible_networks"] / coverage["generated_networks"]
display(coverage)


## View-specific political assortativity

Vote assortativity uses Democratic, Republican, and Other categories. Mutual edges
contribute one incidence in each direction. Matching is always view-specific: directed-out
exposure is matched on directed-out assortativity, and mutual exposure on mutual
assortativity.


In [ ]:
DEMOCRAT_VALUES = {2, "2", "Democrat", "are Democrat", "Joe Biden", "Biden"}
REPUBLICAN_VALUES = {1, "1", "Republican", "are Republican", "Donald Trump", "Trump"}
PARTY_ORDER = ["dem", "rep", "other"]


def normalize_party(value):
    if pd.isna(value):
        return None
    if value in DEMOCRAT_VALUES:
        return "dem"
    if value in REPUBLICAN_VALUES:
        return "rep"
    text = str(value).strip().lower()
    if text in {"democrat", "are democrat", "joe biden", "biden"}:
        return "dem"
    if text in {"republican", "are republican", "donald trump", "trump"}:
        return "rep"
    return "other"


def nominal_vote_assortativity(graph, attribute="w2presvtwho"):
    party = {
        node: normalize_party(data.get(attribute))
        for node, data in graph.nodes(data=True)
    }
    oriented_edges = list(graph.edges())
    if not graph.is_directed():
        oriented_edges = oriented_edges + [(target, source) for source, target in oriented_edges]

    mixing = np.zeros((len(PARTY_ORDER), len(PARTY_ORDER)), dtype=float)
    party_index = {label: index for index, label in enumerate(PARTY_ORDER)}
    for ego, alter in oriented_edges:
        ego_party = party.get(ego)
        alter_party = party.get(alter)
        if ego_party is None or alter_party is None:
            continue
        mixing[party_index[ego_party], party_index[alter_party]] += 1

    total = mixing.sum()
    if total == 0:
        return np.nan, 0
    mixing /= total
    row_marginal = mixing.sum(axis=1)
    column_marginal = mixing.sum(axis=0)
    expected_agreement = np.dot(row_marginal, column_marginal)
    denominator = 1 - expected_agreement
    assortativity = (
        (np.trace(mixing) - expected_agreement) / denominator
        if denominator > 0 else np.nan
    )
    return assortativity, int(total)


graphs_by_view = {}
for network_view, graph_path in GRAPH_INPUTS.items():
    with open(graph_path, "rb") as handle:
        graphs_by_view[network_view] = pickle.load(handle)

assert set(graphs_by_view["directed_out"]) == set(graphs_by_view["mutual"])
network_rows = []
for network_view, models in graphs_by_view.items():
    for model, runs in models.items():
        assert len(runs) == 200
        assert set(runs) == set(graphs_by_view[NETWORK_VIEWS[0]][model])
        for run_id, graph in runs.items():
            assortativity, valid_incidences = nominal_vote_assortativity(graph)
            network_rows.append({
                "network_view": network_view,
                "model": model,
                "model_label": MODEL_LABELS[model],
                "run_id": str(run_id),
                "vote_assortativity": assortativity,
                "n_edges": graph.number_of_edges(),
                "valid_party_edge_incidences": valid_incidences,
            })

network_table = pd.DataFrame(network_rows).merge(
    plausibility_keys, on=["model", "run_id"], how="left", validate="many_to_one"
)
assert len(network_table) == 2_000
assert network_table["plausible_all_5"].notna().all()
assert network_table.groupby("network_view").size().eq(1_000).all()
assert network_table.groupby("network_view")["plausible_all_5"].sum().eq(332).all()
display(
    network_table.groupby(["network_view", "model_label"], observed=True)
    .agg(networks=("run_id", "size"), plausible=("plausible_all_5", "sum"))
    .round(3)
)


## Exposure errors and estimands

For persona \(i\) in network \(A\), the signed error is
\(e_{iA}=\text{LLM category}_{iA}-\text{survey category}_i\).

- `MSD = mean(e_i)` measures subgroup-average directional bias and can cancel.
- `MAE = mean(|e_i|)` measures average absolute inaccuracy within one network.
- `mean(|e_iA − e_iB|)` measures how much the same personas' exposure errors change
  across two matched networks.

The paired-persona metric is the primary outcome in this notebook. Because the survey
category is fixed for a persona, it is also the mean absolute change in that persona's
generated exposure category.


In [ ]:
exposure["llm_cat"] = pd.to_numeric(exposure["llm_cat"], errors="coerce")
exposure["survey_cat"] = pd.to_numeric(exposure["survey_cat"], errors="coerce")
exposure["signed_error"] = exposure["llm_cat"] - exposure["survey_cat"]
exposure["abs_error"] = exposure["signed_error"].abs()
exposure["valid_pair"] = exposure[["llm_cat", "survey_cat"]].notna().all(axis=1)

relation_map = {
    (ego_party, target_party): relation
    for ego_party, target_party, relation in EXPOSURE_RELATIONS
}
relation_exposure = exposure[
    exposure["ego_party"].isin(["dem", "rep", "other"])
    & exposure["party"].isin(["dem", "rep"])
].copy()
relation_exposure["exposure_relation"] = [
    relation_map[(ego_party, target_party)]
    for ego_party, target_party in zip(
        relation_exposure["ego_party"], relation_exposure["party"]
    )
]

run_columns = [
    "network_view", "model", "model_label", "temperature", "method",
    "attr_label", "attribute_order", "run_number", "run_id",
    "exposure_relation",
]
relation_errors = (
    relation_exposure.groupby(run_columns, observed=True, dropna=False)
    .agg(
        n_personas=("caseid", "nunique"),
        n_valid_personas=("valid_pair", "sum"),
        msd=("signed_error", "mean"),
        mae=("abs_error", "mean"),
    ).reset_index()
    .merge(
        network_table[[
            "network_view", "model", "run_id", "vote_assortativity",
            "plausible_all_5", "plausible_metric_count",
        ]],
        on=["network_view", "model", "run_id"],
        how="left", validate="many_to_one",
    )
)
assert len(relation_errors) == 12_000
assert relation_errors["plausible_all_5"].notna().all()
plausible_errors = relation_errors[relation_errors["plausible_all_5"]].copy()
assert plausible_errors.groupby(["network_view", "exposure_relation"]).size().eq(332).all()


## Construct within-model matched network pairs


In [ ]:
def build_within_model_pairs(data, caliper=ASSORTATIVITY_CALIPER):
    frames = []
    configuration_columns = ["temperature", "method", "attr_label"]
    for network_view in NETWORK_VIEWS:
        for relation in RELATION_ORDER:
            relation_data = data[
                data["network_view"].eq(network_view)
                & data["exposure_relation"].eq(relation)
            ]
            for model_label in MODEL_ORDER:
                model_data = relation_data[
                    relation_data["model_label"].eq(model_label)
                ].reset_index(drop=True).copy()
                if model_data.empty:
                    continue
                model_data["configuration"] = (
                    model_data[configuration_columns].astype("string")
                    .fillna("<NA>").agg(" | ".join, axis=1)
                )
                assortativity = model_data["vote_assortativity"].to_numpy()
                delta_assortativity = np.abs(
                    assortativity[:, None] - assortativity[None, :]
                )
                configuration = model_data["configuration"].to_numpy()
                unique_pair = np.triu(
                    np.ones(delta_assortativity.shape, dtype=bool), k=1
                )
                different_configuration = (
                    configuration[:, None] != configuration[None, :]
                )
                left_index, right_index = np.where(
                    unique_pair
                    & different_configuration
                    & (delta_assortativity < caliper)
                )
                if len(left_index) == 0:
                    continue
                msd = model_data["msd"].to_numpy()
                frames.append(pd.DataFrame({
                    "network_view": network_view,
                    "model_label": model_label,
                    "exposure_relation": relation,
                    "run_id_a": model_data["run_id"].to_numpy()[left_index],
                    "run_id_b": model_data["run_id"].to_numpy()[right_index],
                    "configuration_a": configuration[left_index],
                    "configuration_b": configuration[right_index],
                    "assortativity_a": assortativity[left_index],
                    "assortativity_b": assortativity[right_index],
                    "abs_delta_assortativity": delta_assortativity[left_index, right_index],
                    "msd_a": msd[left_index],
                    "msd_b": msd[right_index],
                    "abs_delta_msd": np.abs(msd[left_index] - msd[right_index]),
                }))
    return pd.concat(frames, ignore_index=True)


## Build the 0.02 and 0.05 paper pair sets


In [ ]:
pair_sets = {}
for caliper in [ASSORTATIVITY_CALIPER, SENSITIVITY_CALIPER]:
    label = f"{caliper:.2f}"
    nomination_pairs = build_within_model_pairs(
        relation_errors[relation_errors["network_view"].eq("directed_out")],
        caliper=caliper,
    )
    reciprocal_pairs = build_within_model_pairs(
        plausible_errors[plausible_errors["network_view"].eq("mutual")],
        caliper=caliper,
    )
    assert nomination_pairs["abs_delta_assortativity"].lt(caliper).all()
    assert reciprocal_pairs["abs_delta_assortativity"].lt(caliper).all()
    pair_sets[("directed_out", "all", label)] = nomination_pairs
    pair_sets[("mutual", "plausible", label)] = reciprocal_pairs

nomination_pairs_002_all = pair_sets[("directed_out", "all", "0.02")]
reciprocal_pairs_002_plausible = pair_sets[("mutual", "plausible", "0.02")]
display(pd.DataFrame({
    "pair_set": ["Nomination — all", "Reciprocal — plausible"],
    "relation-level matches": [len(nomination_pairs_002_all), len(reciprocal_pairs_002_plausible)],
}))


## Compare exposure for the same personas in each matched pair


In [ ]:
persona_errors = relation_exposure[
    relation_exposure["valid_pair"]
][[
    "network_view", "run_id", "caseid", "exposure_relation",
    "signed_error", "abs_error",
]].copy()
assert not persona_errors.duplicated([
    "network_view", "run_id", "caseid", "exposure_relation",
]).any()


def build_paired_persona_metrics(pair_frame, caliper_label):
    pair_columns = [
        "network_view", "exposure_relation", "run_id_a", "run_id_b",
    ]
    pair_columns.append("model_label")
    pairs = pair_frame[pair_columns].copy().reset_index(drop=True)
    pairs["pair_id"] = np.arange(len(pairs), dtype=int)

    left_errors = persona_errors.rename(columns={
        "run_id": "run_id_a",
        "signed_error": "signed_error_a",
        "abs_error": "abs_error_a",
    })
    right_errors = persona_errors.rename(columns={
        "run_id": "run_id_b",
        "signed_error": "signed_error_b",
        "abs_error": "abs_error_b",
    })
    paired = pairs.merge(
        left_errors,
        on=["network_view", "exposure_relation", "run_id_a"],
        # A network can be the left member of several matched pairs, so this
        # expansion is intentionally many-to-many; pair_id preserves pair identity.
        how="inner", validate="many_to_many",
    ).merge(
        right_errors,
        on=[
            "network_view", "exposure_relation", "run_id_b", "caseid",
        ],
        how="inner", validate="many_to_one",
    )
    paired["abs_persona_error_difference"] = (
        paired["signed_error_a"] - paired["signed_error_b"]
    ).abs()
    paired["abs_accuracy_magnitude_difference"] = (
        paired["abs_error_a"] - paired["abs_error_b"]
    ).abs()
    paired["error_changed"] = paired["signed_error_a"].ne(
        paired["signed_error_b"]
    )

    metadata_columns = [
        "pair_id", "network_view", "exposure_relation", "run_id_a", "run_id_b",
    ]
    metadata_columns.append("model_label")
    summary = (
        paired.groupby(metadata_columns, observed=True, dropna=False)
        .agg(
            n_common_personas=("caseid", "nunique"),
            mean_error_a=("signed_error_a", "mean"),
            mean_error_b=("signed_error_b", "mean"),
            mean_abs_persona_error_difference=(
                "abs_persona_error_difference", "mean"
            ),
            mean_abs_accuracy_magnitude_difference=(
                "abs_accuracy_magnitude_difference", "mean"
            ),
            share_personas_error_changed=("error_changed", "mean"),
        ).reset_index()
    )
    summary["common_abs_delta_msd"] = (
        summary["mean_error_a"] - summary["mean_error_b"]
    ).abs()
    # Triangle inequality: cancellation can only make |ΔMSD| smaller than the
    # mean absolute paired-persona difference on the same common personas.
    assert (
        summary["mean_abs_persona_error_difference"] + 1e-12
        >= summary["common_abs_delta_msd"]
    ).all()
    summary["caliper"] = caliper_label
    return summary


## Main-text Fig. 4


In [ ]:
eligibility_persona = pd.concat([
    build_paired_persona_metrics(
        nomination_pairs_002_all, "0.02"
    ).assign(eligibility="All runs (1,000)"),
    build_paired_persona_metrics(
        reciprocal_pairs_002_plausible, "0.02"
    ).assign(eligibility="Plausible (332)"),
], ignore_index=True)
assert eligibility_persona["n_common_personas"].gt(0).all()

paper_panel_specs = [
    {
        "letter": "a",
        "network_view": "directed_out",
        "eligibility": "All runs (1,000)",
        "heading": "Nomination networks — All networks",
    },
    {
        "letter": "b",
        "network_view": "mutual",
        "eligibility": "Plausible (332)",
        "heading": "Reciprocal networks — Only plausible",
    },
]

# Verify that panel a uses the complete nomination-network pool, while panel b
# uses only runs whose reciprocal representation passes all five criteria.
nomination_pool = network_table[network_table["network_view"].eq("directed_out")]
reciprocal_plausible_pool = network_table[
    network_table["network_view"].eq("mutual")
    & network_table["plausible_all_5"]
]
assert len(nomination_pool) == 1_000
assert len(reciprocal_plausible_pool) == 332

paper_frames = []
for spec in paper_panel_specs:
    panel = eligibility_persona[
        eligibility_persona["network_view"].eq(spec["network_view"])
        & eligibility_persona["eligibility"].eq(spec["eligibility"])
    ].copy()
    assert not panel.empty
    panel["paper_panel"] = spec["letter"]
    paper_frames.append(panel)
paper_persona = pd.concat(paper_frames, ignore_index=True)
assert set(paper_persona["paper_panel"]) == {"a", "b"}
assert not paper_persona[
    paper_persona["paper_panel"].eq("a")
    & paper_persona["model_label"].eq("Mistral 7B")
].empty
assert paper_persona[
    paper_persona["paper_panel"].eq("b")
    & paper_persona["model_label"].eq("Mistral 7B")
].empty

paper_xmax = max(
    1.0,
    np.ceil(paper_persona["mean_abs_persona_error_difference"].quantile(.99) * 10) / 10,
)
fig, axes = plt.subplots(
    2, len(MODEL_ORDER), figsize=(19, 9.4), sharex=True, sharey=True,
    constrained_layout=False,
)

for row_index, spec in enumerate(paper_panel_specs):
    row_data = paper_persona[paper_persona["paper_panel"].eq(spec["letter"])]
    for column_index, model_label in enumerate(MODEL_ORDER):
        axis = axes[row_index, column_index]
        plot_data = row_data[row_data["model_label"].eq(model_label)]
        if plot_data.empty:
            axis.text(
                .5, .5, "No eligible\nruns", ha="center", va="center",
                transform=axis.transAxes, fontsize=13, color="#555555",
            )
        else:
            sns.boxplot(
                data=plot_data, y="exposure_relation",
                x="mean_abs_persona_error_difference", order=RELATION_ORDER,
                orient="h", whis=(5, 95), showfliers=False, width=.48,
                color=MODEL_COLORS[model_label], saturation=.85, ax=axis,
            )
        axis.set_xlim(0, paper_xmax)
        axis.set_title(model_label if row_index == 0 else "", fontsize=15, pad=10)
        axis.set_xlabel("")
        axis.set_ylabel("")
        axis.set_yticks(
            np.arange(len(RELATION_ORDER)),
            [RELATION_LABELS[relation] for relation in RELATION_ORDER],
        )
        axis.tick_params(axis="both", labelsize=14)
        axis.tick_params(axis="y", labelleft=(column_index == 0))
        axis.grid(axis="y", visible=False)
        sns.despine(ax=axis, left=column_index != 0)

fig.supxlabel("Mean absolute exposure-category difference between networks (same personas)", fontsize=18, y=.035)
fig.subplots_adjust(left=.09, right=.995, bottom=.12, top=.87, hspace=.39, wspace=.13)
fig.canvas.draw()
panel_letter_x = axes[0, 0].get_position().x0 - .055
panel_heading_x = axes[0, -1].get_position().x1
for row_index, spec in enumerate(paper_panel_specs):
    row_top = max(axis.get_position().y1 for axis in axes[row_index])
    fig.text(
        panel_letter_x, row_top + .055, spec["letter"],
        ha="left", va="bottom", fontsize=34, color="black",
    )
    fig.text(
        panel_heading_x, row_top + .060, spec["heading"],
        ha="right", va="bottom", fontsize=20, color="black",
    )

paper_figure_path = (
    IMAGE_DIR / "Fig_4_paired_persona_nominations_all_vs_reciprocal_plausible.pdf"
)
fig.savefig(paper_figure_path, bbox_inches="tight")
plt.show()

panel_counts = pd.DataFrame([
    {
        "panel": "a", "network_view": "directed_out",
        "eligibility_pool": "all", "source_networks": len(nomination_pool),
        "paired_relation_records": len(paper_frames[0]),
    },
    {
        "panel": "b", "network_view": "mutual",
        "eligibility_pool": "plausible reciprocal only",
        "source_networks": len(reciprocal_plausible_pool),
        "paired_relation_records": len(paper_frames[1]),
    },
])
display(panel_counts)
print(f"Saved {paper_figure_path.relative_to(BASE_DIR)}")


## SI Fig. S4 and Table S10


In [ ]:
# Repeat the paper comparison with |Δ vote assortativity| < 0.05.
nomination_pairs_005_all = pair_sets[("directed_out", "all", "0.05")]
reciprocal_pairs_005_plausible = pair_sets[("mutual", "plausible", "0.05")]
assert nomination_pairs_005_all["network_view"].eq("directed_out").all()
assert reciprocal_pairs_005_plausible["network_view"].eq("mutual").all()
assert nomination_pairs_005_all["abs_delta_assortativity"].lt(.05).all()
assert reciprocal_pairs_005_plausible["abs_delta_assortativity"].lt(.05).all()
plausible_reciprocal_ids = set(reciprocal_plausible_pool["run_id"].astype(str))
assert (
    set(reciprocal_pairs_005_plausible["run_id_a"].astype(str))
    | set(reciprocal_pairs_005_plausible["run_id_b"].astype(str))
).issubset(plausible_reciprocal_ids)

nomination_persona_005_all = build_paired_persona_metrics(
    nomination_pairs_005_all, "0.05"
).assign(paper_panel="a")
reciprocal_persona_005_plausible = build_paired_persona_metrics(
    reciprocal_pairs_005_plausible, "0.05"
).assign(paper_panel="b")
paper_persona_005 = pd.concat([
    nomination_persona_005_all, reciprocal_persona_005_plausible,
], ignore_index=True)
assert set(paper_persona_005["paper_panel"]) == {"a", "b"}
assert not paper_persona_005[
    paper_persona_005["paper_panel"].eq("a")
    & paper_persona_005["model_label"].eq("Mistral 7B")
].empty
assert paper_persona_005[
    paper_persona_005["paper_panel"].eq("b")
    & paper_persona_005["model_label"].eq("Mistral 7B")
].empty

# Combine the primary and sensitivity samples so the SI figure compares the
# thresholds directly while otherwise reproducing the main-text analysis.
paper_persona_calipers = pd.concat([
    paper_persona.assign(caliper="0.02"),
    paper_persona_005.assign(caliper="0.05"),
], ignore_index=True)
caliper_order = ["0.02", "0.05"]
caliper_palette = {"0.02": "#176B87", "0.05": "#D97732"}
assert set(paper_persona_calipers["caliper"]) == set(caliper_order)

robustness_xmax = max(
    1.0,
    np.ceil(
        paper_persona_calipers["mean_abs_persona_error_difference"]
        .quantile(.99) * 10
    ) / 10,
)
fig, axes = plt.subplots(
    2, len(MODEL_ORDER), figsize=(19, 9.4), sharex=True, sharey=True,
    constrained_layout=False,
)
for row_index, spec in enumerate(paper_panel_specs):
    row_data = paper_persona_calipers[
        paper_persona_calipers["paper_panel"].eq(spec["letter"])
    ]
    for column_index, model_label in enumerate(MODEL_ORDER):
        axis = axes[row_index, column_index]
        plot_data = row_data[row_data["model_label"].eq(model_label)]
        if plot_data.empty:
            axis.text(
                .5, .5, "No eligible\nruns", ha="center", va="center",
                transform=axis.transAxes, fontsize=13, color="#555555",
            )
        else:
            sns.boxplot(
                data=plot_data, y="exposure_relation",
                x="mean_abs_persona_error_difference", order=RELATION_ORDER,
                hue="caliper", hue_order=caliper_order,
                palette=caliper_palette, dodge=True,
                orient="h", whis=(5, 95), showfliers=False, width=.68,
                saturation=.85, ax=axis,
            )
            if axis.get_legend() is not None:
                axis.get_legend().remove()
        axis.set_xlim(0, robustness_xmax)
        axis.set_title(model_label if row_index == 0 else "", fontsize=15, pad=10)
        axis.set_xlabel("")
        axis.set_ylabel("")
        axis.set_yticks(
            np.arange(len(RELATION_ORDER)),
            [RELATION_LABELS[relation] for relation in RELATION_ORDER],
        )
        axis.tick_params(axis="both", labelsize=14)
        axis.tick_params(axis="y", labelleft=(column_index == 0))
        axis.grid(axis="y", visible=False)
        sns.despine(ax=axis, left=column_index != 0)

fig.supxlabel(
    "Mean absolute exposure-category difference between networks (same personas)",
    fontsize=18, y=.075,
)
legend_handles = [
    Patch(facecolor=caliper_palette[caliper], edgecolor="#555555",
          label=rf"$|\Delta r_{{vote}}| < {caliper}$")
    for caliper in caliper_order
]
fig.legend(
    handles=legend_handles, loc="lower center", bbox_to_anchor=(.5, .005),
    ncol=2, frameon=False, fontsize=14,
)
fig.subplots_adjust(left=.09, right=.995, bottom=.17, top=.87, hspace=.39, wspace=.13)
fig.canvas.draw()
panel_letter_x = axes[0, 0].get_position().x0 - .055
panel_heading_x = axes[0, -1].get_position().x1
for row_index, spec in enumerate(paper_panel_specs):
    row_top = max(axis.get_position().y1 for axis in axes[row_index])
    fig.text(
        panel_letter_x, row_top + .055, spec["letter"],
        ha="left", va="bottom", fontsize=34, color="black",
    )
    fig.text(
        panel_heading_x, row_top + .060, spec["heading"],
        ha="right", va="bottom", fontsize=20, color="black",
    )

robustness_figure_path = (
    IMAGE_DIR
    / "Fig_S4_caliper_002_vs_005_nominations_all_vs_reciprocal_plausible.pdf"
)
fig.savefig(robustness_figure_path, bbox_inches="tight")
fig.savefig(robustness_figure_path.with_suffix(".png"), dpi=300, bbox_inches="tight")
plt.show()

# Counts for SI Table S10. Count unique network pairs rather than the
# six exposure-relation records attached to each matched pair.
count_specs = [
    ("a", "Nomination — all", "0.02", nomination_pairs_002_all),
    ("a", "Nomination — all", "0.05", nomination_pairs_005_all),
    ("b", "Reciprocal — plausible", "0.02", reciprocal_pairs_002_plausible),
    ("b", "Reciprocal — plausible", "0.05", reciprocal_pairs_005_plausible),
]
count_frames = []
for panel, representation, caliper, pair_frame in count_specs:
    unique_pairs = pair_frame.drop_duplicates(
        ["model_label", "run_id_a", "run_id_b"]
    )
    counts = (
        unique_pairs.groupby("model_label", observed=True).size()
        .reindex(MODEL_ORDER, fill_value=0)
        .rename("matched_pairs").rename_axis("model_label").reset_index()
    )
    counts["panel"] = panel
    counts["representation"] = representation
    counts["caliper"] = caliper
    count_frames.append(counts)
robustness_counts = pd.concat(count_frames, ignore_index=True)
robustness_counts = robustness_counts[
    ["panel", "representation", "model_label", "caliper", "matched_pairs"]
]
robustness_counts_path = (
    DATA_DIR / "table_S10_matched_pairs_by_representation_and_caliper_sample_100_v3.csv"
)
robustness_counts.to_csv(robustness_counts_path, index=False)
display(robustness_counts.pivot_table(
    index=["panel", "representation", "model_label"],
    columns="caliper", values="matched_pairs", fill_value=0,
))
print(f"Saved {robustness_figure_path.relative_to(BASE_DIR)}")
